In [2]:
from tqdm import tqdm
import sys
import os
import torch
from torch.utils.data import Dataset, Sampler, DataLoader
import numpy as np
import math
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange, repeat
from einops.layers.torch import Rearrange


In [3]:
# Check PyTorch and device
print(f"PyTorch version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


PyTorch version: 2.10.0+cu128
Device: cuda


In [4]:
REFLECTANCE_SCALE = 10_000.0
CHUNKS_DIR = os.path.expandvars("$HOME/scratch/precomputed_tensors/")

AGRIPOTENTIAL_ABS_DAY = torch.tensor([
       3,   88,  168,  188,  198,  233,  263,  278,
     283,  303,  318,  323,  358,  393,  473,  543,
     553,  573,  583,  603,  628,  663,  728,  733,
     788,  818,  833,  863,  908,  933,  973,  978,
    1028, 1093,
], dtype=torch.long)


In [40]:
class TSViTDataset(Dataset):
    """
    Dataset over precomputed .pt chunks.
 
    Returns per sample:
        data:  (T, C+1, H, W)  float32
        label: (H, W)          long
        doy:   (T,)            long
    """
 
    def __init__(self, mode, add_ndvi=True):
        self.chunk_dir = os.path.join(CHUNKS_DIR, mode)
        self.add_ndvi = add_ndvi
 
        chunk_files = sorted(f for f in os.listdir(self.chunk_dir) if f.endswith(".pt"))
 
        # Build flat index: (filename, within-chunk index)
        self.index = []
        for f in chunk_files:
            path = os.path.join(self.chunk_dir, f)
            # mmap=True just to read shape without loading the whole file
            n = torch.load(path, weights_only=True, mmap=True)["data"].shape[0]
            self.index.extend((f, i) for i in range(n))
 
    def __len__(self):
        return len(self.index)
 
    def __getitem__(self, idx):
        f, patch_idx = self.index[idx]
        path = os.path.join(self.chunk_dir, f)
 
        # mmap=True: the OS loads only the pages we actually touch.
        # For chunked access this is efficient — no full-file load.
        payload = torch.load(path, weights_only=True, mmap=True)
 
        data  = payload["data"][patch_idx].float() / REFLECTANCE_SCALE
        label = payload["label"][patch_idx]
        patch_ids = payload["patch_ids"][patch_idx]
 
        data = data.clamp(0.0, 1.0)   # (T, C, H, W)
 
        if self.add_ndvi:
            # Sentinel-2 band indices (0-based):
            #   band 3 = Red (B04),  band 7 = NIR (B08)
            # Adjust indices if your band ordering differs.
            red  = data[:, 3]                              # (T, H, W)
            nir  = data[:, 7]                              # (T, H, W)
            ndvi = (nir - red) / (nir + red + 1e-6)        # (T, H, W)  range ~[-1, 1]
            ndvi = ndvi.unsqueeze(1)                       # (T, 1, H, W)
            data = torch.cat([data, ndvi], dim=1)          # (T, C+1, H, W)
 
        return data, label, AGRIPOTENTIAL_ABS_DAY, patch_ids


In [6]:
class ChunkAwareSampler(Sampler):
    def __init__(self, dataset, shuffle=True, seed=42):
        self.shuffle = shuffle
        self.rng = torch.Generator().manual_seed(seed)

        chunks = {}
        for i, (f, _) in enumerate(dataset.index):
            chunks.setdefault(f, []).append(i)

        self.chunks = list(chunks.values())

    def __iter__(self):
        order = list(range(len(self.chunks)))
        if self.shuffle:
            order = torch.randperm(len(self.chunks), generator=self.rng).tolist()

        for ci in order:
            indices = self.chunks[ci]
            if self.shuffle:
                perm = torch.randperm(len(indices), generator=self.rng).tolist()
                indices = [indices[j] for j in perm]
            yield from indices

    def __len__(self):
        return sum(len(c) for c in self.chunks)


In [7]:
train_dataset = TSViTDataset("train")
val_dataset   = TSViTDataset("val")

train_sampler = ChunkAwareSampler(train_dataset)
val_sampler   = ChunkAwareSampler(val_dataset, shuffle=False)


In [ ]:
# ---------------------------------------------------------------------------
# TemporalTransformer
#
# Operates independently on each spatial patch across T timesteps.
# num_classes cls tokens are prepended before attention, and only those
# are returned — one per class, each specialised to a different output class.
# Originally used mean pooling here instead (x.mean(dim=1)).
# ---------------------------------------------------------------------------
 
class TemporalTransformer(nn.Module):
    def __init__(self, dim, depth, heads, num_classes, dropout=0.):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=dim * 4,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=depth)
        self.cls_token = nn.Parameter(torch.randn(1, num_classes, dim))
        self.num_classes = num_classes
 
    def forward(self, x):
        B, N, T, D = x.shape
 
        x = x.reshape(B * N, T, D)
        cls = repeat(self.cls_token, '1 k d -> b k d', b=B * N)
        x = torch.cat([cls, x], dim=1)             # [B*N, num_classes+T, dim]
        x = self.encoder(x)[:, :self.num_classes]  # [B*N, num_classes, dim]
 
        return x.reshape(B, N, self.num_classes, D)  # [B, N, K, dim]
 
 
# ---------------------------------------------------------------------------
# SpatialTransformer
#
# Operates independently on each class token across N patch locations.
# Adds a spatial position embedding before attention.
# ---------------------------------------------------------------------------
 
class SpatialTransformer(nn.Module):
    def __init__(self, dim, depth, heads, num_patches, dropout=0.):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=dim * 4,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.encoder         = nn.TransformerEncoder(layer, num_layers=depth)
        self.pos_embedding   = nn.Parameter(torch.randn(1, num_patches, dim))
 
    def forward(self, x):
        # x: [B, N, K, dim]  →  permute so each class is a sequence over N patches
        B, N, K, D = x.shape
 
        x = x.permute(0, 2, 1, 3).reshape(B * K, N, D)    # [B*K, N, dim]
        x = self.encoder(x + self.pos_embedding)          # [B*K, N, dim]
 
        return x.reshape(B, K, N, D)
 
 
# ---------------------------------------------------------------------------
# TSViT
# ---------------------------------------------------------------------------
 
class TSViT(nn.Module):
    """
    Temporal-Spatial ViT for dense pixel-level prediction on SITS.
 
    forward(x, abs_day):
        x:       [B, T, C, H, W]  satellite image time series
        abs_day: [B, T]            days since reference date (e.g. 2017-01-01)
        
        returns: [B, num_classes, H, W]  per-pixel logits
    """
 
    def __init__(
        self,
        in_channels=11,
        num_classes=5,
        img_res=128,
        patch_size=8,
        dim=128,
        temporal_depth=6,
        spatial_depth=2,
        heads=4,
        dropout=0.1,
    ):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches_1d = img_res // patch_size
        N = self.num_patches_1d ** 2
 
        self.patch_embedding = nn.Conv2d(in_channels, dim, kernel_size=patch_size, stride=patch_size)
 
        # Sinusoidal temporal PE — no learned params, computed in forward()
        self.register_buffer('div_term', torch.pow(10000.0, torch.arange(dim // 2).float() / (dim // 2)))
 
        self.temporal = TemporalTransformer(dim, temporal_depth, heads, num_classes, dropout)
        self.spatial = SpatialTransformer(dim, spatial_depth,  heads, N, dropout)
 
        self.decoder = nn.ConvTranspose2d(dim, 1, kernel_size=patch_size, stride=patch_size)
 
    def forward(self, x, abs_day):
        B, T, C, H, W = x.shape
 
        # Sinusoidal temporal PE from absolute day
        arg = abs_day.float().unsqueeze(-1) / self.div_term
        temporal_pe = torch.cat([torch.sin(arg), torch.cos(arg)], dim=-1)  # [B, T, dim]
 
        # Embed each timestep, add temporal PE
        tokens = []
        for t in range(T):
            tok = self.patch_embedding(x[:, t])            # [B, dim, Hp, Wp]
            tok = tok.flatten(2).transpose(1, 2)           # [B, N, dim]
            tok = tok + temporal_pe[:, t].unsqueeze(1)
            tokens.append(tok)
        tokens = torch.stack(tokens, dim=2)                # [B, N, T, dim]
 
        tokens = self.temporal(tokens)                     # [B, N, K, dim]
        tokens = self.spatial(tokens)                      # [B, K, N, dim]
 
        # Decode back to image resolution
        B, K, N, D = tokens.shape
        Hp = Wp = self.num_patches_1d
        tokens = tokens.reshape(B * K, D, Hp, Wp)
        
        return self.decoder(tokens).reshape(B, K, H, W)   # [B, K, H, W]


In [9]:
# ---------------------------------------------------------------------------
# Loss functions
# ---------------------------------------------------------------------------

def ordinal_loss(logits, targets):
    """
    Distance-based ordinal loss.
    logits:  [B, C, H, W]
    targets: [B, H, W]  values in {0..C-1}, -1 = ignored
    """
    mask = targets >= 0
    if mask.sum() == 0:
        return torch.tensor(0.0, device=logits.device)

    # Flatten valid pixels
    logits_valid  = logits.permute(0, 2, 3, 1)[mask]   # [N_valid, C]
    targets_valid = targets[mask]                      # [N_valid]

    probs   = torch.softmax(logits_valid, dim=1)
    classes = torch.arange(logits_valid.shape[1], device=logits.device).float()
    dist    = torch.abs(classes - targets_valid.unsqueeze(1).float())

    return (probs * dist).sum(dim=1).mean()


def loss_fn(logits, targets):
    targets = targets.clone().long() - 1
    ce = F.cross_entropy(logits, targets, ignore_index=-1)
    ord_l = ordinal_loss(logits, targets)
    return ce + 0.3 * ord_l

# ---------------------------------------------------------------------------
# Metric
# ---------------------------------------------------------------------------

def accuracy_pm1(logits, targets):
    """Accuracy with ±1 tolerance (targets still in original 1-indexed space)."""
    preds = torch.argmax(logits, dim=1) + 1   # back to 1..5 space
    mask  = targets > 0

    if mask.sum() == 0:
        return torch.tensor(0.0, device=logits.device)

    diff    = torch.abs(preds[mask] - targets[mask])
    correct = diff <= 1
    return correct.float().mean()


In [10]:
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    total_acc  = 0
    with torch.no_grad():
        for data, label, abs_day in loader:
            data, label, abs_day = data.to(device), label.to(device), abs_day.to(device)
            logits = model(data, abs_day)
            loss   = loss_fn(logits, label)
            total_loss += loss.item()
            total_acc  += accuracy_pm1(logits, label).item()
    return total_loss / len(loader), total_acc / len(loader)


In [41]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    sampler=train_sampler,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,   # keep workers alive between epochs — avoids respawn overhead
    prefetch_factor=2,         # each worker prefetches 2 batches ahead
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    sampler=val_sampler,
    num_workers=4,
    pin_memory=True
)


In [ ]:
# def loss_fn(logits, targets):
#     mask = targets == 0
#     targets = targets.clone()
#     targets = targets - 1
#     targets[mask] = -1

#     return F.cross_entropy(logits, targets, ignore_index=-1)


In [ ]:
model = TSViT().to(device)


/tmp/ipykernel_2205973/47727013.py:17: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=depth)
/tmp/ipykernel_2205973/47727013.py:46: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder         = nn.TransformerEncoder(layer, num_layers=depth)


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.05
)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-4,
    steps_per_epoch=len(train_loader),
    epochs=50,
    pct_start=0.1,    # 10% of training is warmup
)


In [ ]:
print(f"Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB")


Allocated: 0.45 GB
Reserved:  1.02 GB


In [ ]:
torch.cuda.empty_cache()


In [ ]:
epochs             = 150
steps_per_epoch    = len(train_loader)

completed_steps = 80 * steps_per_epoch
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-4,
    total_steps=epochs * steps_per_epoch,
    pct_start=0.1,
    last_epoch=completed_steps - 1,
)


In [ ]:
for epoch in range(100):
    model.train()
    train_loss, train_acc = 0, 0

    for data, label, abs_day, _ in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):
        data, label, abs_day = data.to(device), label.to(device), abs_day.to(device)
        optimizer.zero_grad()
        logits = model(data, abs_day)
        loss = loss_fn(logits, label)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()
        train_loss += loss.item()
        train_acc  += accuracy_pm1(logits, label).item()

    train_loss /= len(train_loader)
    train_acc  /= len(train_loader)
    val_loss, val_acc = evaluate(model, val_loader, device)

    print(f"Epoch {epoch:3d} | train loss {train_loss:.4f} acc {train_acc:.4f} | val loss {val_loss:.4f} acc {val_acc:.4f}")


Epoch   0 | train loss 1.1554 acc 0.8394 | val loss 1.8437 acc 0.7578


Epoch   1 | train loss 1.1544 acc 0.8405 | val loss 1.7457 acc 0.7538


Epoch   2 | train loss 1.1487 acc 0.8406 | val loss 1.8003 acc 0.7557


Epoch   3 | train loss 1.1386 acc 0.8396 | val loss 1.8207 acc 0.7579


Epoch   4 | train loss 1.1346 acc 0.8404 | val loss 1.7396 acc 0.7662


...

Epoch  25 | train loss 1.0071 acc 0.8612 | val loss 1.8977 acc 0.7641


Epoch  26 | train loss 0.9979 acc 0.8618 | val loss 1.9143 acc 0.7653


Epoch  27 | train loss 1.0018 acc 0.8622 | val loss 1.9089 acc 0.7596


Epoch  28 | train loss 0.9935 acc 0.8611 | val loss 1.9327 acc 0.7734


Epoch  29 | train loss 0.9877 acc 0.8623 | val loss 1.9254 acc 0.7644


In [ ]:
save_path = os.path.expandvars("$HOME/scratch/agripotential/tsvit-3.pth")

torch.save({
    'epoch': 100,
    'model': model.state_dict(),
    'optimizer': optimizer.state_dict(),
    'scheduler': scheduler.state_dict(),
}, save_path)


In [10]:
epochs          = 100
steps_per_epoch = len(train_loader)

model = TSViT(
    dim=128,
    temporal_depth=6,
    spatial_depth=2,
    dropout=0.3,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.1,
)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-4,
    total_steps=epochs * steps_per_epoch,
    pct_start=0.1,
)


/tmp/ipykernel_1382354/47727013.py:17: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=depth)
/tmp/ipykernel_1382354/47727013.py:46: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder         = nn.TransformerEncoder(layer, num_layers=depth)


In [ ]:
for epoch in range(50):
    model.train()
    train_loss, train_acc = 0, 0

    for data, label, abs_day, _ in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):
        data, label, abs_day = data.to(device), label.to(device), abs_day.to(device)
        optimizer.zero_grad()
        logits = model(data, abs_day)
        loss = loss_fn(logits, label)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()
        train_loss += loss.item()
        train_acc  += accuracy_pm1(logits, label).item()

    train_loss /= len(train_loader)
    train_acc  /= len(train_loader)
    val_loss, val_acc = evaluate(model, val_loader, device)

    print(f"Epoch {epoch:3d} | train loss {train_loss:.4f} acc {train_acc:.4f} | val loss {val_loss:.4f} acc {val_acc:.4f}")


In [ ]:
save_path = os.path.expandvars("$HOME/scratch/agripotential/tsvit-5.pth")

checkpoint = torch.load(save_path)
model.load_state_dict(checkpoint["model"])
optimizer.load_state_dict(checkpoint["optimizer"])
scheduler.load_state_dict(checkpoint["scheduler"])
start_epoch = checkpoint["epoch"] + 1


In [15]:
save_path = os.path.expandvars("$HOME/scratch/agripotential/tsvit-5.pth")

torch.save({
    'epoch': 100,
    'model': model.state_dict(),
    'optimizer': optimizer.state_dict(),
    'scheduler': scheduler.state_dict(),
}, save_path)


In [59]:
epochs          = 100
steps_per_epoch = len(train_loader)

model = TSViT(
    dim=128,
    temporal_depth=6,
    spatial_depth=2,
    dropout=0.3,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.1,
)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-4,
    total_steps=epochs * steps_per_epoch,
    pct_start=0.1,
)


/tmp/ipykernel_989093/47727013.py:17: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=depth)
/tmp/ipykernel_989093/47727013.py:46: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder         = nn.TransformerEncoder(layer, num_layers=depth)


In [8]:
class TSViTDataset(Dataset):
    """
    Dataset over precomputed .pt chunks.
 
    Returns per sample:
        data:  (T, C+1, H, W)  float32
        label: (H, W)          long
        doy:   (T,)            long
    """
 
    def __init__(self, mode='train', add_ndvi=True):
        self.mode = mode
        self.chunk_dir = os.path.join(CHUNKS_DIR, mode)
        self.add_ndvi = add_ndvi
 
        chunk_files = sorted(f for f in os.listdir(self.chunk_dir) if f.endswith(".pt"))
 
        # Build flat index: (filename, within-chunk index)
        self.index = []
        for f in chunk_files:
            path = os.path.join(self.chunk_dir, f)
            # mmap=True just to read shape without loading the whole file
            n = torch.load(path, weights_only=True, mmap=True)["data"].shape[0]
            self.index.extend((f, i) for i in range(n))
 
    def __len__(self):
        return len(self.index)
 
    def __getitem__(self, idx):
        f, patch_idx = self.index[idx]
        path = os.path.join(self.chunk_dir, f)
 
        # mmap=True: the OS loads only the pages we actually touch.
        # For chunked access this is efficient — no full-file load.
        payload = torch.load(path, weights_only=True, mmap=True)
 
        data  = payload["data"][patch_idx].float() / REFLECTANCE_SCALE
        label = payload["label"][patch_idx]
 
        data = data.clamp(0.0, 1.0)   # (T, C, H, W)
 
        if self.add_ndvi:
            # Sentinel-2 band indices (0-based):
            #   band 3 = Red (B04),  band 7 = NIR (B08)
            # Adjust indices if your band ordering differs.
            red  = data[:, 3]                              # (T, H, W)
            nir  = data[:, 7]                              # (T, H, W)
            ndvi = (nir - red) / (nir + red + 1e-6)        # (T, H, W)  range ~[-1, 1]
            ndvi = ndvi.unsqueeze(1)                       # (T, 1, H, W)
            data = torch.cat([data, ndvi], dim=1)          # (T, C+1, H, W)
        
        if self.mode == 'train':
            # Random horizontal flip
            if torch.rand(1) > 0.5:
                data  = data.flip(-1)
                label = label.flip(-1)
            # Random vertical flip
            if torch.rand(1) > 0.5:
                data  = data.flip(-2)
                label = label.flip(-2)
            # Random 90° rotation
            k = torch.randint(0, 4, (1,)).item()
            data  = torch.rot90(data,  k, dims=(-2, -1))
            label = torch.rot90(label, k, dims=(-2, -1))
 
        return data, label, AGRIPOTENTIAL_ABS_DAY


In [ ]:
best_val_acc    = 0
patience        = 40
no_improve      = 0
val_acc_history = []

for epoch in range(50):
    model.train()
    train_loss, train_acc = 0, 0

    for data, label, abs_day, _ in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):
        data, label, abs_day = data.to(device), label.to(device), abs_day.to(device)
        optimizer.zero_grad()
        logits = model(data, abs_day)
        loss   = loss_fn(logits, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()
        train_acc  += accuracy_pm1(logits, label).item()

    train_loss /= len(train_loader)
    train_acc  /= len(train_loader)
    val_loss, val_acc = evaluate(model, val_loader, device)

    # Smooth val acc over last 10 epochs to reduce oscillation noise
    val_acc_history.append(val_acc)
    smoothed = sum(val_acc_history[-10:]) / min(len(val_acc_history), 10)  # window of 10

    print(f"Epoch {epoch:3d} | train loss {train_loss:.4f} acc {train_acc:.4f} | "
          f"val loss {val_loss:.4f} acc {val_acc:.4f} | smoothed {smoothed:.4f}")

    # Save best model based on smoothed val acc
    if smoothed > best_val_acc:
        best_val_acc = smoothed
        no_improve = 0
        torch.save(model.state_dict(), 'best_model.pt')
        print(f"  ✓ saved best model (smoothed={smoothed:.4f})")
    else:
        no_improve += 1
        print(f"  no improvement for {no_improve}/{patience} epochs")
        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch} — best smoothed val acc: {best_val_acc:.4f}")
            break


In [17]:
save_path = os.path.expandvars("$HOME/scratch/agripotential/tsvit-7.pth")

torch.save({
    'epoch': 60,
    'model': model.state_dict(),
    'optimizer': optimizer.state_dict(),
    'scheduler': scheduler.state_dict(),
}, save_path)


In [ ]:
save_path = os.path.expandvars("$HOME/scratch/agripotential/tsvit-7.pth")

checkpoint = torch.load(save_path)
model.load_state_dict(checkpoint["model"])
optimizer.load_state_dict(checkpoint["optimizer"])
scheduler.load_state_dict(checkpoint["scheduler"])
start_epoch = checkpoint["epoch"] + 1


In [57]:
model = TSViT(
    patch_size=4,
    dim=64,
    temporal_depth=4,
    spatial_depth=2,
    heads=4,
    dropout=0.1,
).to(device)


/tmp/ipykernel_989093/47727013.py:17: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=depth)
/tmp/ipykernel_989093/47727013.py:46: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder         = nn.TransformerEncoder(layer, num_layers=depth)


In [ ]:
optimizer = torch.optim.AdamW([
    {'params': model.temporal.parameters(), 'lr': 1e-4},
    {'params': model.spatial.parameters(),  'lr': 5e-5},
    {'params': model.patch_embedding.parameters(), 'lr': 1e-4},
    {'params': model.decoder.parameters(), 'lr': 1e-4},
], weight_decay=0.05)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=3e-4,
    total_steps=epochs * steps_per_epoch,
    pct_start=0.1,
    div_factor=25,
    final_div_factor=1e4,
)


In [ ]:
best_val_acc    = 0
patience        = 40
no_improve      = 0
val_acc_history = []

for epoch in range(100):
    model.train()
    train_loss, train_acc = 0, 0

    for data, label, abs_day, _ in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):
        data, label, abs_day = data.to(device), label.to(device), abs_day.to(device)
        optimizer.zero_grad()
        logits = model(data, abs_day)
        loss   = loss_fn(logits, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()
        train_acc  += accuracy_pm1(logits, label).item()

    train_loss /= len(train_loader)
    train_acc  /= len(train_loader)
    val_loss, val_acc = evaluate(model, val_loader, device)

    # Smooth val acc over last 5 epochs to reduce oscillation noise
    val_acc_history.append(val_acc)
    smoothed = sum(val_acc_history[-10:]) / min(len(val_acc_history), 10)  # window of 10

    print(f"Epoch {epoch:3d} | train loss {train_loss:.4f} acc {train_acc:.4f} | "
          f"val loss {val_loss:.4f} acc {val_acc:.4f} | smoothed {smoothed:.4f}")

    # Save best model based on smoothed val acc
    if smoothed > best_val_acc:
        best_val_acc = smoothed
        no_improve = 0
        torch.save(model.state_dict(), 'best_model2.pt')
        print(f"  ✓ saved best model (smoothed={smoothed:.4f})")
    else:
        no_improve += 1
        print(f"  no improvement for {no_improve}/{patience} epochs")
        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch} — best smoothed val acc: {best_val_acc:.4f}")
            break


In [60]:
model.load_state_dict(torch.load('best_model.pt', map_location=device))
model.eval()


TSViT(
  (patch_embedding): Conv2d(11, 128, kernel_size=(8, 8), stride=(8, 8))
  (temporal): TemporalTransformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-5): 6 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
          )
          (linear1): Linear(in_features=128, out_features=512, bias=True)
          (dropout): Dropout(p=0.3, inplace=False)
          (linear2): Linear(in_features=512, out_features=128, bias=True)
          (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.3, inplace=False)
          (dropout2): Dropout(p=0.3, inplace=False)
        )
      )
    )
  )
  (spatial): SpatialTransformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
     

In [61]:
evaluate(model, val_loader, device)


(1.8530642311183774, 0.7643589298335873)

In [42]:
import zipfile
from PIL import Image

test_dataset = TSViTDataset(mode='test', add_ndvi=True)
test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

output_dir = os.path.expandvars("$HOME/scratch/agripotential/submissions")
count = 0

with torch.no_grad():
    for data, _, abs_day, patch_ids in test_loader:
        B = data.shape[0]
        data = data.to(device)
        abs_day = abs_day.to(device)

        logits = model(data, abs_day)
        print(f"logits std:  {logits.std():.4f}")    # should be > 0.1
        print(f"logits range: [{logits.min():.3f}, {logits.max():.3f}]")
        print(f"pred classes: {logits.argmax(dim=1).unique(return_counts=True)}")
        preds  = logits.argmax(dim=1) + 1

        for pred, pid in zip(preds.cpu().numpy(), patch_ids):
            img = Image.fromarray(pred.astype(np.uint8), mode='L')
            img.save(os.path.join(output_dir, f"{pid}.png"))
            count += 1

# Zip all predictions
zip_path = f"{output_dir}.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(output_dir)):
        if fname.endswith('.png'):
            zf.write(os.path.join(output_dir, fname), fname)

print(f"Saved {count} predictions → {zip_path}")


logits std:  2.1850
logits range: [-10.054, 6.170]
pred classes: (tensor([0, 1, 2, 3, 4], device='cuda:0'), tensor([49176, 22271, 55734, 43066, 91897], device='cuda:0'))
logits std:  3.4327
logits range: [-11.238, 14.197]
pred classes: (tensor([0, 1, 2, 3, 4], device='cuda:0'), tensor([ 23073,  18976, 116584,   4741,  98770], device='cuda:0'))
logits std:  2.5072
logits range: [-10.216, 12.766]
pred classes: (tensor([0, 1, 2, 3, 4], device='cuda:0'), tensor([ 12341,  74184, 116231,  25826,  33562], device='cuda:0'))
logits std:  2.9164
logits range: [-10.545, 9.887]
pred classes: (tensor([0, 1, 2, 3, 4], device='cuda:0'), tensor([122967,  19718,  75040,  39480,   4939], device='cuda:0'))
logits std:  2.2452
logits range: [-9.926, 7.282]
pred classes: (tensor([0, 1, 2, 3, 4], device='cuda:0'), tensor([38259, 47933, 64354, 48806, 62792], device='cuda:0'))
logits std:  1.6535
logits range: [-7.146, 7.382]
pred classes: (tensor([0, 1, 2, 3, 4], device='cuda:0'), tensor([87300,  1684, 48232